# Experiments 3 + 4 — MovieLens: paper figures

This notebook only loads `results/exp3_3`, `results/exp4_baseline`, and `results/exp4_amplified_imbalance` and plots — it never recomputes algorithm results (see CLAUDE.md and `docs/experiments.md` §0.4). Generate the inputs with `make exp3`, `make exp4` and `make exp4-acb`.

Experiment 3 and Experiment 4 are separate pipelines (different environments and parallelism; see `docs/experiments.md`) but are presented together as **three settings of one story**:
1. **Setting 1 (Experiment 3, `exp3_3`):** K=2, N=8, real random movie pairs. Algorithms: `kabc`, `acb`, `tas-fw`, `unif-kkm`.
2. **Setting 2 (Experiment 4, `baseline`):** K=7, N=40, imbalanced prototypes. Algorithms: `kabc`, `unif-kkm`, `acb` (`tas-fw` is infeasible at this scale).
3. **Setting 3 (Experiment 4, `amplified_imbalance`):** K=12, N=76, more imbalanced. Same three algorithms.

Output:
- **Main figure** (`results/figures/exp3_exp4_combined_row`): one row of five panels — setting-1 example pair, setting-1 ARI, setting-1 wall-clock, setting-1 ARI vs. pair separation, and settings 2–3 ARI per algorithm.

- **Many-cluster headline histograms** (`results/figures/exp4_fig0_histograms_baseline`, `..._amplified_imbalance`): real rating histograms of the prototype movies drawn at rep 0 for the two many-cluster settings. (The exp3 K=2 example-pair histogram is kept commented out; panel 1 of the combined row shows it.)

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

sys.path.insert(0, "..")
from kabc.plotting import set_style, savefig, ALGO_COLORS, ALGO_LABELS

set_style()

# (setting label, results path)
SETTINGS = [
    ("exp3_3", "../results/exp3_3/results.parquet"),
    ("exp4_baseline", "../results/exp4_baseline/results.parquet"),
    ("exp4_amplified_imbalance", "../results/exp4_amplified_imbalance/results.parquet"),
]

dfs = {}
for label, path in SETTINGS:
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(
            f"{p} not found. Run `python -m scripts.run_experiment3` / `run_experiment4` first -- "
            "this notebook only plots existing results, it never recomputes them."
        )
    dfs[label] = pd.read_parquet(p)
    print(f"loaded {len(dfs[label])} rows ({dfs[label]['rep'].nunique()} reps) from {p}")

dfs["exp3_3"].head()

## Three example histograms (one figure per setting, saved separately)

Real rating histograms, not recomputed from any algorithm output. All three settings use the SAME illustration style (`plot_prototype_histograms`, originally Experiment 4's): one bar-group per real prototype movie drawn at `rep=0`, grouped by rating value, role/count labeled on the x-axis. Setting 1 (`exp3_3`) is just the K=2 case of this (roles "A"/"B").

In [ ]:
# --- Shared histogram-by-rating illustration style (originally Experiment 4's) ---
RATING_VALUES = [1, 2, 3, 4, 5]
RATING_COLORS = [plt.cm.Greys(v) for v in (0.2, 0.4, 0.6, 0.8, 1.0)]  # grayscale, as in the combined row


def plot_prototype_histograms(ratings_df, title, prototypes: list[dict], fig_name: str):
    K = len(prototypes)
    fig, ax = plt.subplots(figsize=(1.7 * K + 2.3, 4.2))
    bar_width = 0.8 / len(RATING_VALUES)
    x_base = np.arange(K)

    for r_idx, rating in enumerate(RATING_VALUES):
        heights = []
        for p in prototypes:
            pool = ratings_df.loc[ratings_df["movie_id"] == p["movie_id"], "rating"]
            heights.append(float((pool == rating).mean()))
        offset = (r_idx - (len(RATING_VALUES) - 1) / 2) * bar_width
        ax.bar(x_base + offset, heights, bar_width * 0.92, color=RATING_COLORS[r_idx],
               label=f"rating {rating}")

    ax.set_xticks(x_base)
    labels = [f"{p['title']}\n({p['role']}, n={p['count']})" for p in prototypes]
    ax.set_xticklabels(labels, rotation=28, ha="right", fontsize=7)
    ax.set_ylabel("proportion of ratings")
    ax.set_title(title, fontsize=9)
    ax.legend(title="rating", fontsize=7, title_fontsize=7, frameon=False,
              loc="center left", bbox_to_anchor=(1.01, 0.5))
    fig.tight_layout()
    savefig(fig, fig_name, fig_dir="../results/figures")
    plt.show()

In [ ]:
# Many-cluster headline histograms (exp4 settings 2/3). The exp3 K=2 example pair is not plotted here:
# the combined row's panel 1 already shows it.
# # --- Setting 1 (exp3_3): movie-pair A/B, as a K=2 case of the same style ---
# with open("../configs/exp3.yaml") as f:
#     exp3_config = yaml.safe_load(f)
#
# ratings3 = pd.read_parquet("../data/ml-1m-207/ratings.parquet")
# HEADLINE_REP = 0
# row = dfs["exp3_3"][dfs["exp3_3"]["rep"] == HEADLINE_REP].iloc[0]
# exp3_prototypes = [
#     {"movie_id": row["movie_id_a"], "title": row["title_a"], "role": "A", "count": exp3_config["arms_per_prototype"]},
#     {"movie_id": row["movie_id_b"], "title": row["title_b"], "role": "B", "count": exp3_config["arms_per_prototype"]},
# ]
# plot_prototype_histograms(
#     ratings3,
#     f"Setting 1 (exp3_3): real sampled movie pair, rep={HEADLINE_REP}\n"
#     rf"$|\bar\mu_A-\bar\mu_B|$={abs(row['mean_a'] - row['mean_b']):.2f}, MMD$^2$={row['mmd2_ab']:.3f} (K=2)",
#     exp3_prototypes,
#     "exp3_fig0_headline_pair",
# )
#

# --- Settings 2/3 (exp4 baseline / amplified_imbalance): prototype histograms, rep=0 ---
from scripts.run_experiment4 import _load_movie_pool, _choose_resampled_prototypes

with open("../configs/exp4.yaml") as f:
    exp4_config = yaml.safe_load(f)

ratings4 = pd.read_parquet("../data/ml-1m-207/ratings.parquet")
movie_ids4, means4, titles4, pools4, mmd24 = _load_movie_pool(Path("..") / exp4_config["data_dir"])

EXP4_SETTINGS = ["baseline", "amplified_imbalance"]
EXP4_SETTING_TITLES = {
    "baseline": "Setting 2: baseline (K=7, far 12:2, close/far mean-gap-resampled every rep)",
    "amplified_imbalance": "Setting 3: amplified_imbalance (K=12, far 20:2, same resampling mechanism)",
}
EXP4_HEADLINE_REP = 0
EXP4_REP_SEED = exp4_config["seed_multiplier"] * exp4_config["exp_id"] + EXP4_HEADLINE_REP

for setting in EXP4_SETTINGS:
    env, roles, pair_info = _choose_resampled_prototypes(
        exp4_config["conditions"][setting], EXP4_REP_SEED, movie_ids4, means4, titles4, pools4,
    )
    protos = [
        {"movie_id": mid, "title": titles4[mid], "role": role, "count": count}
        for mid, role, count in zip(env.movie_ids, roles, env.counts)
    ]

    logged = dfs[f"exp4_{setting}"][dfs[f"exp4_{setting}"]["rep"] == EXP4_HEADLINE_REP].iloc[0]
    derived_by_role = {p["role"]: p["movie_id"] for p in protos}
    assert derived_by_role["close_a"] == logged["close_a_movie_id"]
    assert derived_by_role["close_b"] == logged["close_b_movie_id"]
    assert derived_by_role["far_large"] == logged["far_large_movie_id"]
    assert derived_by_role["far_small"] == logged["far_small_movie_id"]

    plot_prototype_histograms(
        ratings4,
        f"{EXP4_SETTING_TITLES[setting]}\nreal rating histograms, rep={EXP4_HEADLINE_REP} (K={len(protos)})",
        protos,
        f"exp4_fig0_histograms_{setting}",
    )

## Combined one-row figure (5 panels)

1. Setting 1 headline example: a hand-picked sampled movie pair with a small mean gap but non-trivial MMD² (rep 35).
2. Setting 1: mean ARI per algorithm.
3. Setting 1: mean wall-clock per rep (log scale).
4. Setting 1: ARI vs. pair separation (MMD²), one point per rep; crosses = did not converge (hit the budget cap).
5. Settings 2-3 (exp4): KABC vs. Unif-KKM mean ARI.


In [ ]:
ratings3 = pd.read_parquet("../data/ml-1m-207/ratings.parquet")
EXP3_ALGOS = ["kabc", "acb", "tas-fw", "unif-kkm"]
EXP4_ALGOS = ["kabc", "unif-kkm", "acb"]
EXP4_BAR_SETTINGS = [
    ("exp4_baseline", "(K = 7, N = 40)"),
    ("exp4_amplified_imbalance", "(K = 12, N = 76)"),
]


def _bar_stats(s, robust, clip01):
    """Center and (lower, upper) error-bar lengths: mean +/- std, or median with IQR if robust."""
    if robust:
        c, lo, hi = s.median(), s.quantile(0.25), s.quantile(0.75)
    else:
        c, lo, hi = s.mean(), s.mean() - s.std(), s.mean() + s.std()
    if clip01:
        lo, hi = max(lo, 0.0), min(hi, 1.0)
    return c, [[max(c - lo, 0.0)], [max(hi - c, 0.0)]]


def single_setting_bar(ax, df, algos, metric_col, ylabel, log_scale=False, robust=False, clip01=False):
    """One group (one setting), one bar per algorithm."""
    n_algo = len(algos)
    bar_width = 0.8 / n_algo
    for a_idx, algo in enumerate(algos):
        c, err = _bar_stats(df[df["algorithm"] == algo][metric_col], robust, clip01)
        x = (a_idx - (n_algo - 1) / 2) * bar_width
        ax.bar(x, c, yerr=err, width=bar_width * 0.92, capsize=2,
               color=ALGO_COLORS[algo], alpha=0.88, label=ALGO_LABELS[algo])
    ax.set_xticks([])
    ax.set_ylabel(ylabel)
    if log_scale:
        ax.set_yscale("log")
    if clip01:
        ax.set_ylim(0, 1.05)


def multi_setting_bar(ax, settings, algos, metric_col, ylabel):
    """One group per setting, one bar per algorithm within each group."""
    n_algo = len(algos)
    bar_width = 0.8 / n_algo
    group_gap = 0.6
    xtick_positions, xtick_labels = [], []
    for g_idx, (label, title) in enumerate(settings):
        df = dfs[label]
        group_center = g_idx * (1.0 + group_gap)
        for a_idx, algo in enumerate(algos):
            c, err = _bar_stats(df[df["algorithm"] == algo][metric_col], False, True)
            x = group_center + (a_idx - (n_algo - 1) / 2) * bar_width
            ax.bar(x, c, yerr=err, width=bar_width * 0.92, capsize=2,
                   color=ALGO_COLORS[algo], alpha=0.88, label=ALGO_LABELS[algo])
        xtick_positions.append(group_center)
        xtick_labels.append(title)
    ax.set_xticks(xtick_positions)
    ax.set_xticklabels(xtick_labels, fontsize=8)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, 1.05)

df_exp3 = dfs["exp3_3"]
APPENDIX_MARKERS = {"kabc": "o", "tas-fw": "D", "unif-kkm": "s"}

# Headline pair for this figure: picked by hand from the 100 sampled pairs -- small mean gap
# (|mean_A - mean_B| = 0.01) but non-trivial MMD^2 (0.011), so the two movies differ in shape, not mean.
COMBINED_ROW_REP = 35
_hrow = df_exp3[df_exp3["rep"] == COMBINED_ROW_REP].iloc[0]
row_prototypes = [
    {"movie_id": _hrow["movie_id_a"], "title": _hrow["title_a"], "role": "A"},
    {"movie_id": _hrow["movie_id_b"], "title": _hrow["title_b"], "role": "B"},
]


TITLE_KW = dict(fontsize=10, fontweight="bold")
LABEL_FS, TICK_FS = 9, 8
GRAY_RATING_COLORS = [plt.cm.Greys(v) for v in (0.2, 0.4, 0.6, 0.8, 1.0)]

import textwrap
fig = plt.figure(figsize=(13, 2.9), layout="constrained")
sf_left, sf_mid, sf_right = fig.subfigures(1, 3, width_ratios=[1.3, 3.3, 1.15], wspace=0.02)
ax_hist = sf_left.subplots()
ax_ari, ax_wall, ax_scat = sf_mid.subplots(1, 3, gridspec_kw={"width_ratios": [1, 1, 1.3]})
ax_exp4 = sf_right.subplots()

# Panel 1: headline pair histogram (hand-picked pair, see COMBINED_ROW_REP)
ax = ax_hist
bw = 0.8 / len(RATING_VALUES)
xb = np.arange(len(row_prototypes))
for r_idx, rating in enumerate(RATING_VALUES):
    hs = [float((ratings3.loc[ratings3["movie_id"] == pr["movie_id"], "rating"] == rating).mean())
          for pr in row_prototypes]
    ax.bar(xb + (r_idx - (len(RATING_VALUES) - 1) / 2) * bw, hs, bw * 0.92,
           color=GRAY_RATING_COLORS[r_idx], edgecolor="0.3", linewidth=0.4, label=str(rating))
ax.set_xticks(xb)
ax.set_xticklabels([textwrap.fill(pr["title"], 16) for pr in row_prototypes], fontsize=7)
ax.set_ylabel("proportion of ratings", fontsize=LABEL_FS)
ax.set_ylim(0, 0.52)
ax.legend(title="rating", fontsize=7, title_fontsize=7, frameon=False, ncol=5, loc="upper center",
          columnspacing=0.8, handlelength=0.8)
sf_left.suptitle(rf"2-Cluster (Example pair:" + "\n" + rf"$\Delta$ = {abs(_hrow['mean_a'] - _hrow['mean_b']):.2f}, MMD$^2$ = {_hrow['mmd2_ab']:.3f})",
                 fontsize=9, fontweight="bold")

# Panels 2-4: Setting 1 comparison
single_setting_bar(ax_ari, df_exp3, EXP3_ALGOS, "ari", "mean ARI ± std", clip01=True)
single_setting_bar(ax_wall, df_exp3, EXP3_ALGOS, "wall_clock_seconds", "median wall-clock/rep (s) (IQR)", log_scale=True, robust=True)

ax = ax_scat
rng_jit = np.random.default_rng(0)
for algo in ["unif-kkm", "tas-fw", "kabc"]:
    sub_all = df_exp3[df_exp3["algorithm"] == algo]
    for stopped in (True, False):
        sub = sub_all[sub_all["stopped"] == stopped]
        jit = -rng_jit.uniform(0, 0.03, len(sub))
        if stopped:
            ax.scatter(sub["mmd2_ab"], sub["ari"] + jit, s=22, marker=APPENDIX_MARKERS[algo],
                       color=ALGO_COLORS[algo], alpha=0.8)
        else:
            ax.scatter(sub["mmd2_ab"], sub["ari"] + jit, s=32, marker="x",
                       color=ALGO_COLORS[algo], linewidths=1.3, alpha=0.95)
ax.set_xscale("log")
ax.set_xlabel(r"pair separation, MMD$^2$")
ax.set_ylabel("ARI")
ax.set_ylim(-0.25, 1.05)
sf_mid.suptitle("Comparison of Algorithms in 2-Cluster setting", **TITLE_KW)

# Panel 5: exp4
multi_setting_bar(ax_exp4, EXP4_BAR_SETTINGS, EXP4_ALGOS, "ari", "mean ARI ± std")
sf_right.suptitle("Many Clusters", **TITLE_KW)

for a in (ax_hist, ax_ari, ax_wall, ax_scat, ax_exp4):
    a.xaxis.label.set_size(LABEL_FS)
    a.yaxis.label.set_size(LABEL_FS)
    a.tick_params(axis="y", labelsize=TICK_FS)
    a.tick_params(axis="x", labelsize=TICK_FS)
ax_hist.tick_params(axis="x", labelsize=7)
ax_exp4.tick_params(axis="x", labelsize=TICK_FS)

handles = [plt.Line2D([], [], marker="s", linestyle="", color=ALGO_COLORS[a], label=ALGO_LABELS[a])
           for a in EXP3_ALGOS]
handles.append(plt.Line2D([], [], marker="x", linestyle="", color="0.3", label="did not converge (hit budget cap)"))
fig.legend(handles=handles, loc="outside lower center", ncol=len(handles), frameon=False, fontsize=8)
savefig(fig, "exp3_exp4_combined_row", fig_dir="../results/figures")
plt.show()
